# DataCoolie Databricks Polars Sample

Secondary Databricks sample for file + delta scenarios using `PolarsEngine`.

Run this after the Spark sample succeeds on your workspace paths.

In [ ]:
%pip install "datacoolie[databricks-polars]" --upgrade
# Optional local wheel install:
# %pip install /Volumes/workspace/default/datacoolie_sim/libraries/datacoolie-0.1.0-py3-none-any.whl --force-reinstall

In [ ]:
dbutils.library.restartPython()  # type: ignore[name-defined]

In [ ]:
from datacoolie.core.models import DataCoolieRunConfig
from datacoolie.engines.polars_engine import PolarsEngine
from datacoolie.metadata.file_provider import FileProvider
from datacoolie.orchestration.driver import DataCoolieDriver
from datacoolie.platforms.databricks_platform import DatabricksPlatform

# Replace with your UC volume root if needed.
VOLUME_ROOT = "/Volumes/workspace/default/datacoolie_sim"
METADATA_PATH = f"{VOLUME_ROOT}/metadata/databricks_use_cases.json"
BASE_LOG_PATH = f"{VOLUME_ROOT}/logs/datacoolie"

# Narrow first run for smoke validation. Use STAGE="" to run all file+delta scenarios.
STAGE = STAGE = "read_file,write_file,transform_schema,transform_dedup,transform_addcols,source_backward,date_folder_dest,date_folder_src,hive_partition_src,transform_schema,transform_dedup,transform_schema,source_backward,streaming_deferred"

platform = DatabricksPlatform()
storage_options = {}
engine = PolarsEngine(platform=platform, storage_options=storage_options)
metadata = FileProvider(config_path=METADATA_PATH, platform=platform)
run_config = DataCoolieRunConfig(max_workers=8)

with DataCoolieDriver(
    engine=engine,
    metadata_provider=metadata,
    base_log_path=BASE_LOG_PATH,
    config=run_config,
) as driver:
    result = driver.run(stage=STAGE)

print(
    "Result:",
    {
        "total": result.total,
        "succeeded": result.succeeded,
        "failed": result.failed,
        "skipped": result.skipped,
    },
)